<a href="https://colab.research.google.com/github/yunapark17/2026-capstone-design/blob/main/%EC%88%98%EC%A0%95%EC%9A%A9_%EC%BA%A1%EC%8A%A4%ED%86%A4%EB%94%94%EC%9E%90%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1번 셀: 준비
##### 데이터 다운로드, 불러오기,
##### 테스트/트레인 분리
##### 결측치 정리는 교차검증 Pipeline에 위임


# 1. SECOM 데이터 다운로드
!wget -q https://archive.ics.uci.edu/static/public/179/secom.zip -O secom.zip
!unzip -oq secom.zip -d secom_data

import pandas as pd
from sklearn.model_selection import train_test_split

# 2. 데이터 불러오기
X = pd.read_csv('secom_data/secom.data', sep=' ', header=None)
y = pd.read_csv('secom_data/secom_labels.data',
                sep=' ', header=None,
                names=['label', 'timestamp'])

y_label = y['label']

print("원본 데이터 크기:", X.shape)
print("Pass/Fail 분포:")
print(y_label.value_counts())

# 3. 먼저 Train/Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y_label,
    test_size=0.2,
    random_state=42,
    stratify=y_label
)

# 결측률/상수 열 제거와 대체값 계산은 2번 셀의 Pipeline에서 수행합니다.
# 따라서 각 교차검증 Fold의 학습 데이터만으로 전처리를 학습합니다.

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

print("\nTrain 라벨 분포:")
print(y_train.value_counts())

print("\nTest 라벨 분포:")
print(y_test.value_counts())


[2번 셀]
1. 기존 모델 5개와 임계값 탐색 범위는 유지
2. 기본 전처리 / 결측 여부 추가 / 변수 200개·전체 사용 / 상관계수 0.95 초과 중복 열 제거 비교
3. 결측 열 제거·중앙값·상관관계·변수 선택은 각 학습 Fold에서 계산
4. 총 70개 조합 × 5-Fold (350회 Fold 학습), 동일 전처리는 임시 캐시로 재사용


In [ ]:
#@title 2번 셀
# ==========================================
# 2번 셀: 여러 머신러닝 모델 비교
# ==========================================

!pip install -q xgboost imbalanced-learn

import numpy as np
import pandas as pd
from functools import partial
from tempfile import TemporaryDirectory
from joblib import Memory
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.feature_selection import (
    SelectFromModel, VarianceThreshold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, cross_val_predict
)
from sklearn.metrics import (
    precision_score, recall_score,
    fbeta_score, average_precision_score
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# ------------------------------------------
# 1. XGBoost를 위해 라벨을 0/1로 변환
# 기존 -1 = 정상, 1 = 불량
# 변경  0 = 정상, 1 = 불량
# ------------------------------------------

y_train_binary = (y_train == 1).astype(int)

# 불량 대비 정상 데이터 비율
class_ratio = (
    (y_train_binary == 0).sum()
    / (y_train_binary == 1).sum()
)

print("정상/불량 비율:", round(class_ratio, 2))


# ------------------------------------------
# 2. 모델 생성 함수
# ------------------------------------------

# 전처리는 fit 때 학습한 열/대체값을 transform 때 그대로 사용합니다.
class SecomPreprocessor(TransformerMixin, BaseEstimator):
    def __init__(self, add_indicator=False, max_missing=0.5):
        self.add_indicator = add_indicator
        self.max_missing = max_missing

    def fit(self, X, y=None):
        frame = pd.DataFrame(X)
        self.n_features_in_ = frame.shape[1]
        self.columns_ = frame.columns.to_numpy()
        keep = (frame.isna().mean() <= self.max_missing) & (frame.nunique(dropna=True) > 1)
        self.numeric_columns_ = frame.columns[keep].tolist()
        # 값 자체를 제외한 열도 결측 여부는 신호일 수 있어 별도로 보존합니다.
        missing = frame.isna().any() & frame.notna().any()
        self.indicator_columns_ = frame.columns[missing].tolist() if self.add_indicator else []
        if not self.numeric_columns_ and not self.indicator_columns_:
            raise ValueError("학습에 사용할 비상수 변수가 없습니다.")
        self.imputer_ = None
        if self.numeric_columns_:
            self.imputer_ = SimpleImputer(strategy='median', keep_empty_features=True)
            self.imputer_.fit(frame[self.numeric_columns_])
        return self

    def transform(self, X):
        check_is_fitted(self, 'columns_')
        frame = pd.DataFrame(X)
        if not np.array_equal(frame.columns.to_numpy(), self.columns_):
            raise ValueError("학습 데이터와 입력 열의 순서가 다릅니다.")
        numeric = (self.imputer_.transform(frame[self.numeric_columns_])
                   if self.imputer_ is not None else np.empty((len(frame), 0)))
        flags = frame[self.indicator_columns_].isna().to_numpy(dtype=float)
        return pd.DataFrame(np.column_stack([numeric, flags]), index=frame.index,
                            columns=self.get_feature_names_out())

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'columns_')
        return np.array([f'sensor_{c}' for c in self.numeric_columns_] +
                        [f'missing_sensor_{c}' for c in self.indicator_columns_], dtype=object)


class CorrelationFilter(TransformerMixin, BaseEstimator):
    def __init__(self, threshold=None):
        self.threshold = threshold

    def fit(self, X, y=None):
        frame = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(frame.columns, dtype=object)
        self.n_features_in_ = frame.shape[1]
        self.support_ = np.ones(frame.shape[1], dtype=bool)
        if self.threshold is not None:
            if not 0 <= self.threshold <= 1:
                raise ValueError("상관계수 기준은 0~1 사이여야 합니다.")
            corr = frame.corr().abs().to_numpy()
            retained = []
            # 앞서 남긴 열과 중복되는 열만 제거하는 결정적인 순서입니다.
            for index in range(frame.shape[1]):
                if retained and np.any(corr[index, retained] > self.threshold):
                    self.support_[index] = False
                else:
                    retained.append(index)
        return self

    def transform(self, X):
        check_is_fitted(self, 'support_')
        frame = pd.DataFrame(X)
        if not np.array_equal(frame.columns.to_numpy(), self.feature_names_in_):
            raise ValueError("학습 데이터와 입력 열의 순서가 다릅니다.")
        return frame.iloc[:, self.support_]

    def get_support(self):
        check_is_fitted(self, 'support_')
        return self.support_.copy()

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'support_')
        names = self.feature_names_in_ if input_features is None else np.asarray(input_features)
        return names[self.support_]


def capped_feature_count(X, limit):
    return min(limit, X.shape[1])


# 같은 Fold·같은 전처리의 중간 결과를 재사용해 반복 학습을 줄입니다.
# 임시 캐시는 저장소 밖에 생성되며 다음 실행에서 다시 만들 수 있습니다.
pipeline_cache_dir = TemporaryDirectory(prefix='secom-preprocessing-')
pipeline_memory = Memory(pipeline_cache_dir.name, verbose=0)

def make_model(top_n, method, preprocessing='baseline'):
    if preprocessing not in {'baseline', 'missing_flags', 'expanded_features', 'correlation'}:
        raise ValueError("알 수 없는 전처리: " + preprocessing)

    steps = [
        ('preprocess', SecomPreprocessor(add_indicator=preprocessing == 'missing_flags')),
        ('variance', VarianceThreshold().set_output(transform='pandas')),
        ('correlation', CorrelationFilter(threshold=0.95 if preprocessing == 'correlation' else None)),
    ]

    if top_n is not None:
        steps.append(('selector', SelectFromModel(
            RandomForestClassifier(
                n_estimators=120, min_samples_leaf=2,
                class_weight='balanced_subsample', random_state=42, n_jobs=2
            ),
            threshold=-np.inf,
            max_features=partial(capped_feature_count, limit=top_n)
        )))
    else:
        steps.append(('selector', 'passthrough'))
    steps.append(('scaler', StandardScaler()))

    # SMOTE 방식: 불량을 정상의 50%까지 증가
    if method.endswith('SMOTE05'):
        steps.append(
            ('smote', SMOTE(
                sampling_strategy=0.5,
                k_neighbors=3,
                random_state=42
            ))
        )

    # Random Forest
    if method.startswith('RF'):

        weight = (
            'balanced_subsample'
            if method == 'RF_weight'
            else None
        )

        clf = RandomForestClassifier(
            n_estimators=250,
            min_samples_leaf=2,
            class_weight=weight,
            random_state=42,
            n_jobs=2
        )

    # XGBoost
    elif method.startswith('XGB'):

        weight = (
            class_ratio
            if method == 'XGB_weight'
            else 1
        )

        clf = XGBClassifier(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=3,
            reg_lambda=5,
            scale_pos_weight=weight,
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            random_state=42,
            n_jobs=2
        )

    # Logistic Regression
    elif method == 'LOGIT_weight':

        clf = LogisticRegression(
            C=0.1,
            class_weight='balanced',
            max_iter=2000,
            random_state=42
        )

    else:
        raise ValueError("알 수 없는 모델 방식: " + method)

    steps.append(('classifier', clf))

    return Pipeline(steps, memory=pipeline_memory)


# ------------------------------------------
# 3. 비교할 학습 방법 및 변수 개수
# ------------------------------------------

methods = [
    'RF_weight',
    'RF_SMOTE05',
    'XGB_weight',
    'XGB_SMOTE05',
    'LOGIT_weight'
]

# 한 번에 한 가지 전처리 변경의 효과를 비교합니다.
preprocessing_options = {
    'baseline': [10, 20, 50, 100],
    'missing_flags': [10, 20, 50, 100],
    'expanded_features': [200, None],  # None은 변수 선택 없이 전체 사용
    'correlation': [10, 20, 50, 100],
}

# 5-Fold 교차검증
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Threshold 0.02~0.80
thresholds = np.round(
    np.arange(0.02, 0.801, 0.01), 2
)

results = []


# ------------------------------------------
# 4. 모델별 교차검증 및 Threshold 비교
# ------------------------------------------

for preprocessing, feature_counts in preprocessing_options.items():
    for method in methods:

        for top_n in feature_counts:

            print(f"학습 중: {preprocessing}, {method}, Top {top_n}")

            model = make_model(top_n, method, preprocessing)

            # Train 내부 5-Fold 검증
            # 각 샘플이 검증용일 때의 예측 점수
            oof_proba = cross_val_predict(
                model,
                X_train,
                y_train_binary,
                cv=cv,
                method='predict_proba',
                n_jobs=1
            )[:, 1]

            # Threshold와 무관한 AP 평가
            ap = average_precision_score(
                y_train_binary,
                oof_proba
            )

            # Threshold별 성능 비교
            for t in thresholds:

                pred = (oof_proba >= t).astype(int)

                precision = precision_score(
                    y_train_binary,
                    pred,
                    zero_division=0
                )

                recall = recall_score(
                    y_train_binary,
                    pred,
                    zero_division=0
                )

                f2 = fbeta_score(
                    y_train_binary,
                    pred,
                    beta=2,
                    zero_division=0
                )

                results.append({
                    'preprocessing': preprocessing,
                    'method': method,
                    'top_n': 'all' if top_n is None else top_n,
                    'threshold': float(t),
                    'precision': precision,
                    'recall': recall,
                    'F2': f2,
                    'AP': ap
                })


# ------------------------------------------
# 5. 전체 비교 결과 저장
# ------------------------------------------

results_df = pd.DataFrame(results)

print("\n모든 교차검증 완료!")

# 모델 조합별 AP 비교
ap_comparison = (
    results_df
    .sort_values('AP', ascending=False)
    .drop_duplicates(['preprocessing', 'method', 'top_n'])
    [['preprocessing', 'method', 'top_n', 'AP']]
)

print("\n모델별 Average Precision 비교")
display(ap_comparison.round(4))


[3번 셀]
1. 정밀도 하한을 20%에서 10%로 완화하고 교차검증 검출률 목표를 70%로 설정
2. 두 조건을 만족하는 후보 중 정밀도가 가장 높은 설정 선택
3. 조건 미달이면 자동 완화하지 않고 중단
4. Test 결과로 임계값을 재선택하지 않으며, Test 검출률 50% 달성 여부와 오탐 수를 함께 평가


In [ ]:
#@title 3번 셀: 검출률 우선 목표를 만족하는 후보 선택
# 교차검증에서 설정을 고정한 뒤 Test에서 50% 달성 여부를 평가합니다.
# 아래 조건은 교차검증 기준이며 Test 성능을 보장하지 않습니다.
target_recall = 0.70
min_precision = 0.10


def rank_settings(results_df, target_recall=0.70, min_precision=0.10):
    """교차검증 검출률 목표·정밀도 하한을 만족하는 후보 중 정밀도 최대화."""
    if not 0 <= target_recall <= 1 or not 0 <= min_precision <= 1:
        raise ValueError("검출률 목표와 정밀도 하한은 0~1 사이여야 합니다.")
    candidates = results_df[
        (results_df['recall'] >= target_recall) &
        (results_df['precision'] >= min_precision)
    ].copy()
    return candidates.sort_values(
        by=['precision', 'recall', 'AP'],
        ascending=[False, False, False],
        kind='stable'
    )


# 재실행 시 조건 미달이면 과거 모델을 실수로 평가하지 않도록 초기화합니다.
best_model = None
best_threshold = None
comparison_models = {}
ranked = rank_settings(results_df, target_recall, min_precision)
if ranked.empty:
    raise ValueError(
        f"교차검증 검출률 {target_recall:.0%}·정밀도 {min_precision:.0%} 조건을 만족하는 후보가 없습니다. "
        "Test 결과를 보면서 임계값을 낮추지 말고 학습 데이터에서 기준을 검토하세요."
    )

best = ranked.iloc[0]
best_method = best['method']
best_top_n = None if best['top_n'] == 'all' else int(best['top_n'])
best_threshold = float(best['threshold'])
best_preprocessing = best['preprocessing']

# 각 전처리 방식에서도 같은 기준으로 후보를 Test 평가 전에 고정합니다.
choices = []
for preprocessing, group in results_df.groupby('preprocessing', sort=False):
    eligible = rank_settings(group, target_recall, min_precision)
    if eligible.empty:
        print(f"{preprocessing}: 교차검증 조건 미달로 비교에서 제외")
        continue
    choices.append(eligible.iloc[0])
comparison_choices = pd.DataFrame(choices).reset_index(drop=True)

print(f"교차검증 Recall >= {target_recall:.0%}, Precision >= {min_precision:.0%}")
print("조건을 만족하는 후보 중 정밀도가 가장 높은 설정을 선택했습니다.")
display(comparison_choices.round(4))
print("\n전체 최선 설정:", best_preprocessing, best_method, best_top_n, best_threshold)
print(f"교차검증 검출률: {best['recall']:.2%}")
print(f"교차검증 정밀도: {best['precision']:.2%}")

for row in comparison_choices.itertuples(index=False):
    top_n = None if row.top_n == 'all' else int(row.top_n)
    fitted = make_model(top_n, row.method, row.preprocessing)
    fitted.fit(X_train, y_train_binary)
    comparison_models[row.preprocessing] = fitted
best_model = comparison_models[best_preprocessing]
print("최종 학습 완료: 4번 셀에서 테스트 검출률과 정상 오탐을 확인하세요.")


[4번 셀]


In [ ]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    PrecisionRecallDisplay
)
import matplotlib.pyplot as plt

# 1. Test 데이터의 불량 예측 점수
test_proba = best_model.predict_proba(X_test)[:, 1]

# 2. 선택한 Threshold로 최종 판정
y_pred_final = np.where(
    test_proba >= best_threshold,
    1, -1
)

# 3. 혼동행렬
print("최종 혼동행렬")
final_cm = confusion_matrix(y_test, y_pred_final, labels=[-1, 1])
print(final_cm)
tn, fp, fn, tp = final_cm.ravel()
test_recall = recall_score(y_test, y_pred_final, pos_label=1)
test_precision = precision_score(y_test, y_pred_final, pos_label=1, zero_division=0)
print(f"테스트 검출률: {test_recall:.2%} ({tp}/{tp + fn}개)")
print(f"테스트 정밀도: {test_precision:.2%}")
print(f"정상 제품 오탐: {fp}/{tn + fp}개 ({fp / (tn + fp):.2%})")
print("테스트 검출률 50% 초과:", "달성" if test_recall > 0.50 else "미달")

# 4. 전체 성능
print("\n최종 성능 평가")
print(classification_report(
    y_test, y_pred_final,
    labels=[-1, 1],
    digits=4,
    zero_division=0
))

# 5. Fail 클래스 주요 성능
print("Fail Recall:",
      round(recall_score(y_test, y_pred_final, pos_label=1), 4))

print("Fail Precision:",
      round(precision_score(
          y_test, y_pred_final,
          pos_label=1, zero_division=0
      ), 4))

print("Fail F1:",
      round(f1_score(
          y_test, y_pred_final,
          pos_label=1, zero_division=0
      ), 4))

print("Average Precision:",
      round(average_precision_score(
          (y_test == 1).astype(int),
          test_proba
      ), 4))

# 6. 결측 표시 변수도 포함하여 실제 선택된 변수 이름 확인
selected_names = best_model.named_steps['preprocess'].get_feature_names_out()
selected_names = selected_names[best_model.named_steps['variance'].get_support()]
selected_names = selected_names[best_model.named_steps['correlation'].get_support()]
selector = best_model.named_steps['selector']
if not isinstance(selector, str):
    selected_names = selected_names[selector.get_support()]
print("\n최종 선택 변수 (missing_sensor_*는 결측 여부):")
print(selected_names.tolist())

# 7. Precision-Recall 곡선
PrecisionRecallDisplay.from_predictions(
    (y_test == 1).astype(int),
    test_proba
)

plt.title(f"{best_preprocessing} / {best_method} - Precision Recall Curve")
plt.grid(True)
plt.show()

# 8. 교차검증에서 미리 고정한 전처리별 후보의 Test 결과
# 이 표를 보고 모델·전처리·임계값을 다시 선택하지 않습니다.
test_rows = []
for row in comparison_choices.itertuples(index=False):
    scores = comparison_models[row.preprocessing].predict_proba(X_test)[:, 1]
    predicted = np.where(scores >= row.threshold, 1, -1)
    tn, fp, fn, tp = confusion_matrix(y_test, predicted, labels=[-1, 1]).ravel()
    test_rows.append({
        'preprocessing': row.preprocessing, 'method': row.method,
        'top_n': row.top_n, 'threshold': row.threshold,
        'cv_recall': row.recall, 'cv_precision': row.precision,
        'test_recall': recall_score(y_test, predicted, pos_label=1),
        'test_precision': precision_score(y_test, predicted, pos_label=1, zero_division=0),
        'detected': int(tp), 'missed': int(fn), 'false_alarms': int(fp),
        'false_positive_rate': fp / (tn + fp),
        'test_AP': average_precision_score((y_test == 1).astype(int), scores)
    })
test_comparison = pd.DataFrame(test_rows)
print("\n전처리별 고정 후보의 Test 비교 (Test 기준 재선택 금지)")
display(test_comparison.round(4))
